# 🏎️ RAG Testing — F1 Race Strategist AI

This notebook validates that the retrieval pipeline returns **relevant context**
for strategy-related questions.

Modules under test:
- `rag.embedder` — ChromaDB client & collection access
- `rag.retriever` — semantic search + FastF1 enrichment

In [ ]:
import sys, os, logging
# Ensure project root is on sys.path
sys.path.insert(0, os.path.abspath(".."))
logging.basicConfig(level=logging.WARNING)  # keep notebook output clean

## 1. Embedder — Collection Health Check

In [ ]:
from rag.embedder import get_collection, list_collections

# List all collections
collections = list_collections()
print("📦 Collections in ChromaDB:")
for c in collections:
    print(f"  • {c['name']}: {c['count']} documents")

# Verify f1_circuits exists
col = get_collection("f1_circuits")
assert col.count() > 0, "f1_circuits is empty — run `python -m rag.ingestor` first!"
print(f"\n✅ f1_circuits: {col.count()} documents")

## 2. Circuit Vector Search

Test that semantic search returns the correct circuits for various queries.

In [ ]:
from rag.retriever import retrieve_circuits

test_queries = [
    "street circuit in Monaco",
    "high speed track in Italy",
    "circuit in Silverstone England",
    "race track in São Paulo Brazil",
    "desert circuit in Bahrain",
]

for query in test_queries:
    results = retrieve_circuits(query, n_results=3)
    top = results[0]
    print(f"🔍 \"{query}\"")
    for r in results:
        print(f"   [{r['distance']:.4f}] {r['metadata']['name']} ({r['metadata']['country']})")
    print()

## 3. Filtered Search (metadata `where` clause)

ChromaDB supports metadata filtering. Let's test country-scoped searches.

In [ ]:
# Search only UK circuits
uk_results = retrieve_circuits(
    "fast circuit",
    n_results=5,
    where_filter={"country": "UK"},
)
print("🇬🇧 UK circuits matching 'fast circuit':")
for r in uk_results:
    print(f"   [{r['distance']:.4f}] {r['metadata']['name']} — {r['metadata']['location']}")

## 4. Full Race Context — Silverstone 2022

This is the key test: **"qual estratégia venceu em Silverstone 2022?"**

The retriever should return:
- Correct circuit (Silverstone)
- Race results with Sainz P1
- Winner's stint strategy (MEDIUM → MEDIUM → HARD → SOFT)
- Weather data (rain at Silverstone!)

In [ ]:
from rag.retriever import retrieve_race_context

ctx = retrieve_race_context(
    query="qual estratégia venceu em Silverstone 2022?",
    year=2022,
    circuit="Silverstone",
)

if ctx["error"]:
    print(f"⚠️ Error: {ctx['error']}")
else:
    print(ctx["context_text"])

In [ ]:
# Validate key facts
assert ctx["error"] is None, f"Retrieval failed: {ctx['error']}"

results_df = ctx["race_results"]
winner = results_df.iloc[0]["Abbreviation"]
print(f"🏆 Winner: {winner}")
assert winner == "SAI", f"Expected SAI, got {winner}"

stints = ctx["winner_stints"]
compounds = list(stints["Compound"])
print(f"🔧 Strategy: {' → '.join(compounds)}")
assert len(compounds) >= 3, f"Expected 3+ stints, got {len(compounds)}"

weather = ctx["weather"]
assert not weather.empty, "Weather data should not be empty"
print(f"🌧️ Weather points: {len(weather)}")

print("\n✅ All assertions passed!")

## 5. Additional Race Contexts

Test with different races to confirm robustness.

In [ ]:
test_races = [
    {"query": "Monaco GP strategy", "year": 2023, "circuit": "Monaco"},
    {"query": "Monza race tactics", "year": 2023, "circuit": "Monza"},
]

for race in test_races:
    print("=" * 60)
    ctx = retrieve_race_context(**race)
    if ctx["error"]:
        print(f"⚠️ {race['circuit']} {race['year']}: {ctx['error']}")
    else:
        results_df = ctx["race_results"]
        winner = results_df.iloc[0]["Abbreviation"]
        stints = ctx["winner_stints"]
        compounds = list(stints["Compound"])
        print(f"🏁 {race['circuit']} {race['year']}")
        print(f"   Winner: {winner}")
        print(f"   Strategy: {' → '.join(compounds)}")
        print(f"   Context: {len(ctx['context_text'])} chars")
    print()

## 6. Context Text Quality

Check that the assembled context string contains all required sections.

In [ ]:
ctx = retrieve_race_context(
    query="strategy at Silverstone",
    year=2022,
    circuit="Silverstone",
)

text = ctx["context_text"]
required_sections = [
    "Race Context",
    "Race Results",
    "Winning Strategy",
    "Top-5 Strategies",
    "Weather",
]

print("📋 Context structure validation:")
for section in required_sections:
    found = section in text
    status = "✅" if found else "❌"
    print(f"   {status} {section}")
    assert found, f"Missing section: {section}"

print(f"\n✅ All {len(required_sections)} required sections present!")
print(f"📏 Total context length: {len(text)} chars")